# آموزش نهایی YOLO26s-P2 برای تشخیص انسان در تصاویر هوایی

این نوت‌بوک مستقل و قابل اجرای مستقیم در Google Colab است و مراحل زیر را انجام می‌دهد:

- قفل‌کردن نسخه Ultralytics
- اتصال Google Drive و بررسی GPU
- یافتن خودکار دیتاست Person-only
- ساخت معماری رسمی YOLO26s-P2
- انتقال وزن‌های سازگار از `yolo26s.pt`
- آموزش سه‌مرحله‌ای ۸+۳۲+۱۰ اپوک
- Resume امن هر مرحله
- انتخاب Checkpoint بر اساس Validation
- ارزیابی نهایی یک‌باره روی Test
- تحلیل تشخیصی Recall بر اساس اندازه هدف
- Export به ONNX FP32 و FP16
- تولید گزارش CSV، JSON و SHA256

> مجموعه Test برای انتخاب مدل استفاده نمی‌شود؛ فقط مدل نهایی روی Test ارزیابی می‌شود.

In [ ]:
# ============================================================
# CELL 1 — Install pinned dependencies
# ============================================================

%pip install -q --upgrade \
    ultralytics==8.4.114 \
    onnx \
    onnxslim \
    onnxruntime-gpu \
    pyyaml \
    pandas \
    tqdm

In [ ]:
# ============================================================
# CELL 2 — Imports, Google Drive and environment report
# ============================================================

from __future__ import annotations

from pathlib import Path
from datetime import datetime
from collections import Counter, defaultdict
from typing import Any, Iterable
import gc
import hashlib
import json
import platform
import shutil
import subprocess
import sys
import warnings

import cv2
import numpy as np
import pandas as pd
import torch
import yaml
from tqdm.auto import tqdm
from IPython.display import display

import ultralytics
from ultralytics import YOLO

from google.colab import drive
drive.mount("/content/drive")

print("=" * 100)
print("ENVIRONMENT")
print("=" * 100)
print("Python:", sys.version.replace("\n", " "))
print("Ultralytics:", ultralytics.__version__)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("CUDA runtime:", torch.version.cuda)

if not torch.cuda.is_available():
    raise RuntimeError("Enable a GPU runtime in Colab before training.")

GPU_INDEX = 0
GPU_NAME = torch.cuda.get_device_name(GPU_INDEX)
GPU_MEMORY_GB = torch.cuda.get_device_properties(GPU_INDEX).total_memory / 1024**3

print("GPU:", GPU_NAME)
print(f"GPU memory: {GPU_MEMORY_GB:.2f} GB")
subprocess.run(["nvidia-smi"], check=False)

assert ultralytics.__version__ == "8.4.114", (
    "Unexpected Ultralytics version. Restart the runtime and rerun CELL 1."
)

In [ ]:
# ============================================================
# CELL 3 — Global configuration
# ============================================================

LOCAL_PROJECT_ROOT = Path("/content/aerial_person_final_product")
DRIVE_PROJECT_ROOT = Path(
    "/content/drive/MyDrive/aerial_person_yolo26s_p2_final"
)

CONFIG_DIR = DRIVE_PROJECT_ROOT / "configs"
RUNS_DIR = DRIVE_PROJECT_ROOT / "runs"
REPORTS_DIR = DRIVE_PROJECT_ROOT / "reports"
EXPORTS_DIR = DRIVE_PROJECT_ROOT / "exports"
FINAL_DIR = DRIVE_PROJECT_ROOT / "final_model"

for directory in (CONFIG_DIR, RUNS_DIR, REPORTS_DIR, EXPORTS_DIR, FINAL_DIR):
    directory.mkdir(parents=True, exist_ok=True)

FULL_DATA_CANDIDATES = [
    LOCAL_PROJECT_ROOT / "datasets/visdrone_person/data.yaml",
    LOCAL_PROJECT_ROOT / "datasets/visdrone_person_full/data.yaml",
]

HYBRID_DATA_CANDIDATES = [
    LOCAL_PROJECT_ROOT / "datasets/visdrone_person_context_tiles/data.yaml",
    LOCAL_PROJECT_ROOT / "datasets/visdrone_person_hybrid/data.yaml",
]

BASE_WEIGHTS = "yolo26s.pt"
SEED = 42
DEVICE = 0
WORKERS = 4
NBS = 64
MAX_DET = 1000
VAL_IOU = 0.70
TEST_CONF = 0.001
IMAGE_SIZE_FINAL = 1280

STAGE_1_EPOCHS = 8
STAGE_2_EPOCHS = 32
STAGE_3_EPOCHS = 10

FORCE_RESTART_STAGES = False
RUN_SIZE_DIAGNOSTIC = True
RUN_ONNX_EXPORT = True
RUN_ONNX_VALIDATION = True

# Optional baseline checkpoint. Leave as None to skip.
BASELINE_PT = None

print("Drive project root:", DRIVE_PROJECT_ROOT)

In [ ]:
# ============================================================
# CELL 4 — Dataset discovery and validation
# ============================================================

IMAGE_SUFFIXES = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}


def first_existing(paths: Iterable[Path]) -> Path | None:
    for path in paths:
        if path.exists():
            return path.resolve()
    return None


def load_yaml(path: Path) -> dict[str, Any]:
    with path.open("r", encoding="utf-8") as handle:
        data = yaml.safe_load(handle)
    if not isinstance(data, dict):
        raise ValueError(f"Invalid YAML dictionary: {path}")
    return data


def dataset_root_from_yaml(yaml_path: Path, config: dict[str, Any]) -> Path:
    root_value = config.get("path")
    if root_value is None:
        return yaml_path.parent.resolve()
    root = Path(str(root_value))
    return root if root.is_absolute() else (yaml_path.parent / root).resolve()


def resolve_split_entries(yaml_path: Path, split_name: str) -> list[Path]:
    config = load_yaml(yaml_path)
    root = dataset_root_from_yaml(yaml_path, config)
    value = config.get(split_name)
    if value is None:
        return []
    values = value if isinstance(value, list) else [value]
    output = []
    for item in values:
        path = Path(str(item))
        output.append(path if path.is_absolute() else (root / path).resolve())
    return output


def collect_images(entries: list[Path]) -> list[Path]:
    images = []
    for entry in entries:
        if entry.is_file() and entry.suffix.lower() == ".txt":
            for line in entry.read_text(encoding="utf-8").splitlines():
                candidate = Path(line.strip())
                if not candidate.is_absolute():
                    candidate = (entry.parent / candidate).resolve()
                if candidate.suffix.lower() in IMAGE_SUFFIXES:
                    images.append(candidate)
        elif entry.is_file() and entry.suffix.lower() in IMAGE_SUFFIXES:
            images.append(entry)
        elif entry.is_dir():
            images.extend(
                path for path in entry.rglob("*")
                if path.is_file() and path.suffix.lower() in IMAGE_SUFFIXES
            )
    return sorted(set(images))


def image_to_label_path(image_path: Path) -> Path:
    parts = list(image_path.parts)
    for index in range(len(parts) - 1, -1, -1):
        if parts[index] == "images":
            parts[index] = "labels"
            return Path(*parts).with_suffix(".txt")
    return image_path.with_suffix(".txt")


def inspect_dataset(yaml_path: Path, title: str) -> dict[str, Any]:
    config = load_yaml(yaml_path)
    required = ("train", "val", "test")
    missing = [key for key in required if not config.get(key)]
    if missing:
        raise ValueError(f"{title} is missing splits {missing}: {yaml_path}")

    nc = int(config.get("nc", len(config.get("names", []))))
    if nc != 1:
        warnings.warn(
            f"{title} reports nc={nc}. Verify that this is the intended Person-only dataset."
        )

    result = {
        "title": title,
        "yaml": str(yaml_path),
        "nc": nc,
        "names": config.get("names"),
    }

    for split in required:
        images = [
            path for path in collect_images(resolve_split_entries(yaml_path, split))
            if path.exists()
        ]
        if not images:
            raise FileNotFoundError(
                f"No images were found for split '{split}' in {yaml_path}"
            )
        labels = [image_to_label_path(path) for path in images]
        result[f"{split}_images"] = len(images)
        result[f"{split}_labels"] = sum(path.exists() for path in labels)
        result[f"{split}_backgrounds"] = sum(
            (not path.exists()) or path.stat().st_size == 0 for path in labels
        )
    return result


FULL_DATA_YAML = first_existing(FULL_DATA_CANDIDATES)
HYBRID_DATA_YAML = first_existing(HYBRID_DATA_CANDIDATES)

if HYBRID_DATA_YAML is None:
    raise FileNotFoundError(
        "Hybrid/context data.yaml was not found:\n"
        + "\n".join(str(path) for path in HYBRID_DATA_CANDIDATES)
    )

if FULL_DATA_YAML is None:
    warnings.warn(
        "A separate full-scene data.yaml was not found. "
        "The hybrid data.yaml will be used in all stages."
    )
    FULL_DATA_YAML = HYBRID_DATA_YAML

dataset_summary_df = pd.DataFrame([
    inspect_dataset(FULL_DATA_YAML, "Full-scene dataset"),
    inspect_dataset(HYBRID_DATA_YAML, "Hybrid/context dataset"),
])
display(dataset_summary_df)

dataset_summary_path = REPORTS_DIR / "dataset_summary.csv"
dataset_summary_df.to_csv(dataset_summary_path, index=False)

print("Full data YAML:", FULL_DATA_YAML)
print("Hybrid data YAML:", HYBRID_DATA_YAML)

In [ ]:
# ============================================================
# CELL 5 — Conservative automatic batch selection
# ============================================================

def choose_batches(memory_gb: float) -> dict[str, int]:
    if memory_gb >= 20:
        return {"stage1": 8, "stage2": 4, "stage3": 4, "val": 4}
    if memory_gb >= 14:
        return {"stage1": 4, "stage2": 2, "stage3": 2, "val": 2}
    if memory_gb >= 8:
        return {"stage1": 2, "stage2": 1, "stage3": 1, "val": 1}
    return {"stage1": 1, "stage2": 1, "stage3": 1, "val": 1}


BATCHES = choose_batches(GPU_MEMORY_GB)
print(json.dumps(BATCHES, indent=2))

In [ ]:
# ============================================================
# CELL 6 — Prepare official YOLO26s-P2 and transfer report
# ============================================================

PACKAGE_ROOT = Path(ultralytics.__file__).resolve().parent
OFFICIAL_P2_YAML = PACKAGE_ROOT / "cfg/models/26/yolo26-p2.yaml"

if not OFFICIAL_P2_YAML.exists():
    raise FileNotFoundError(
        f"Official yolo26-p2.yaml was not found: {OFFICIAL_P2_YAML}"
    )

P2_YAML = CONFIG_DIR / "yolo26s-p2.yaml"
p2_config = load_yaml(OFFICIAL_P2_YAML)
p2_config["nc"] = 1

with P2_YAML.open("w", encoding="utf-8") as handle:
    yaml.safe_dump(p2_config, handle, sort_keys=False, allow_unicode=True)

base_model = YOLO(BASE_WEIGHTS)
fresh_p2_model = YOLO(str(P2_YAML))

base_state = base_model.model.state_dict()
p2_state = fresh_p2_model.model.state_dict()

compatible_keys = [
    key for key, value in p2_state.items()
    if key in base_state and tuple(value.shape) == tuple(base_state[key].shape)
]
compatible_numel = sum(p2_state[key].numel() for key in compatible_keys)
total_numel = sum(value.numel() for value in p2_state.values())

transfer_report = {
    "base_weights": BASE_WEIGHTS,
    "p2_yaml": str(P2_YAML),
    "compatible_tensor_keys": len(compatible_keys),
    "total_p2_tensor_keys": len(p2_state),
    "compatible_numel": int(compatible_numel),
    "total_p2_numel": int(total_numel),
    "compatible_numel_ratio": float(compatible_numel / total_numel),
}

transfer_report_path = REPORTS_DIR / "pretrained_transfer_report.json"
transfer_report_path.write_text(
    json.dumps(transfer_report, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

print(json.dumps(transfer_report, indent=2))
fresh_p2_model.info(detailed=False, verbose=True)

del base_model, fresh_p2_model
gc.collect()
torch.cuda.empty_cache()

In [ ]:
# ============================================================
# CELL 7 — Safe training and resume utilities
# ============================================================

def clear_memory() -> None:
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def count_completed_epochs(results_csv: Path) -> int:
    if not results_csv.exists():
        return 0
    try:
        return len(pd.read_csv(results_csv))
    except Exception:
        return 0


def stage_paths(stage_name: str) -> dict[str, Path]:
    run_dir = RUNS_DIR / stage_name
    return {
        "run_dir": run_dir,
        "last": run_dir / "weights/last.pt",
        "best": run_dir / "weights/best.pt",
        "results": run_dir / "results.csv",
    }


def run_or_resume_stage(
    *,
    stage_name: str,
    source: str | Path,
    epochs: int,
    data_yaml: Path,
    train_args: dict[str, Any],
    build_p2_from_base: bool = False,
) -> Path:
    paths = stage_paths(stage_name)

    if FORCE_RESTART_STAGES and paths["run_dir"].exists():
        shutil.rmtree(paths["run_dir"])

    completed = count_completed_epochs(paths["results"])

    print("\n" + "=" * 110)
    print("STAGE:", stage_name)
    print("Expected epochs:", epochs)
    print("Completed rows:", completed)
    print("Data:", data_yaml)
    print("=" * 110)

    if paths["best"].exists() and paths["last"].exists() and completed >= epochs:
        print("Stage already complete. Reusing best.pt.")
        return paths["best"]

    clear_memory()

    if paths["last"].exists() and completed > 0:
        print("Incomplete stage detected. Resuming last.pt.")
        model = YOLO(str(paths["last"]))
        model.train(resume=True)
    else:
        print("Starting new stage from:", source)
        model = (
            YOLO(str(P2_YAML)).load(str(source))
            if build_p2_from_base
            else YOLO(str(source))
        )
        model.train(
            data=str(data_yaml),
            epochs=epochs,
            project=str(RUNS_DIR),
            name=stage_name,
            exist_ok=True,
            **train_args,
        )

    paths = stage_paths(stage_name)
    selected = paths["best"] if paths["best"].exists() else paths["last"]

    if not selected.exists():
        raise FileNotFoundError(f"No checkpoint was produced for {stage_name}")

    del model
    clear_memory()
    print("Selected checkpoint:", selected)
    return selected


COMMON_ARGS = {
    "device": DEVICE,
    "workers": WORKERS,
    "single_cls": True,
    "optimizer": "AdamW",
    "amp": True,
    "cos_lr": True,
    "seed": SEED,
    "deterministic": True,
    "max_det": MAX_DET,
    "iou": VAL_IOU,
    "nbs": NBS,
    "weight_decay": 5e-4,
    "momentum": 0.937,
    "patience": 100,
    "save": True,
    "save_period": 5,
    "plots": True,
    "val": True,
    "verbose": True,
    "cache": False,
    "rect": False,
    "compile": False,
    "channels_last": False,
}

In [ ]:
# ============================================================
# CELL 8 — Stage 1: P2 warm-up at 960
# ============================================================

stage1_args = {
    **COMMON_ARGS,
    "imgsz": 960,
    "batch": BATCHES["stage1"],
    "freeze": 10,
    "lr0": 2.0e-4,
    "lrf": 0.10,
    "warmup_epochs": 1.0,
    "warmup_momentum": 0.8,
    "warmup_bias_lr": 0.05,
    "mosaic": 1.0,
    "close_mosaic": 0,
    "mixup": 0.03,
    "copy_paste": 0.0,
    "cutmix": 0.0,
    "hsv_h": 0.015,
    "hsv_s": 0.50,
    "hsv_v": 0.35,
    "degrees": 5.0,
    "translate": 0.10,
    "scale": 0.45,
    "shear": 2.0,
    "perspective": 0.0002,
    "fliplr": 0.50,
    "flipud": 0.0,
    "erasing": 0.30,
    "auto_augment": "randaugment",
}

STAGE_1_BEST = run_or_resume_stage(
    stage_name="stage1_p2_warmup_960",
    source=BASE_WEIGHTS,
    epochs=STAGE_1_EPOCHS,
    data_yaml=FULL_DATA_YAML,
    train_args=stage1_args,
    build_p2_from_base=True,
)

print("Stage 1 best:", STAGE_1_BEST)

In [ ]:
# ============================================================
# CELL 9 — Stage 2: Main hybrid training at 1280
# ============================================================

stage2_args = {
    **COMMON_ARGS,
    "imgsz": 1280,
    "batch": BATCHES["stage2"],
    "freeze": None,
    "lr0": 4.0e-4,
    "lrf": 0.05,
    "warmup_epochs": 1.0,
    "warmup_momentum": 0.8,
    "warmup_bias_lr": 0.05,
    "mosaic": 1.0,
    "close_mosaic": 0,
    "mixup": 0.03,
    "copy_paste": 0.0,
    "cutmix": 0.0,
    "hsv_h": 0.015,
    "hsv_s": 0.50,
    "hsv_v": 0.35,
    "degrees": 5.0,
    "translate": 0.10,
    "scale": 0.45,
    "shear": 2.0,
    "perspective": 0.0002,
    "fliplr": 0.50,
    "flipud": 0.0,
    "erasing": 0.40,
    "auto_augment": "randaugment",
}

STAGE_2_BEST = run_or_resume_stage(
    stage_name="stage2_hybrid_1280",
    source=STAGE_1_BEST,
    epochs=STAGE_2_EPOCHS,
    data_yaml=HYBRID_DATA_YAML,
    train_args=stage2_args,
)

print("Stage 2 best:", STAGE_2_BEST)

In [ ]:
# ============================================================
# CELL 10 — Stage 3: Natural fine-tuning without Mosaic
# ============================================================

stage3_args = {
    **COMMON_ARGS,
    "imgsz": 1280,
    "batch": BATCHES["stage3"],
    "freeze": None,
    "lr0": 5.0e-5,
    "lrf": 0.10,
    "warmup_epochs": 0.5,
    "warmup_momentum": 0.8,
    "warmup_bias_lr": 0.02,
    "mosaic": 0.0,
    "close_mosaic": 0,
    "mixup": 0.0,
    "copy_paste": 0.0,
    "cutmix": 0.0,
    "hsv_h": 0.010,
    "hsv_s": 0.30,
    "hsv_v": 0.25,
    "degrees": 2.0,
    "translate": 0.05,
    "scale": 0.20,
    "shear": 0.0,
    "perspective": 0.0,
    "fliplr": 0.50,
    "flipud": 0.0,
    "erasing": 0.10,
    "auto_augment": "randaugment",
}

STAGE_3_BEST = run_or_resume_stage(
    stage_name="stage3_final_finetune_1280",
    source=STAGE_2_BEST,
    epochs=STAGE_3_EPOCHS,
    data_yaml=FULL_DATA_YAML,
    train_args=stage3_args,
)

print("Stage 3 best:", STAGE_3_BEST)

In [ ]:
# ============================================================
# CELL 11 — Finalize checkpoint, SHA256 and metadata
# ============================================================

def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


FINAL_PT = FINAL_DIR / "yolo26s_p2_person_1280_final_best.pt"
shutil.copy2(STAGE_3_BEST, FINAL_PT)

FINAL_SHA256 = sha256_file(FINAL_PT)
FINAL_SIZE_MB = FINAL_PT.stat().st_size / 1024**2

final_model = YOLO(str(FINAL_PT))
final_model.info(detailed=False, verbose=True)

parameter_count = sum(p.numel() for p in final_model.model.parameters())

model_metadata = {
    "model_name": "YOLO26s-P2 Person-only",
    "architecture_yaml": str(P2_YAML),
    "final_checkpoint": str(FINAL_PT),
    "sha256": FINAL_SHA256,
    "size_mb": FINAL_SIZE_MB,
    "parameters": int(parameter_count),
    "input_size": IMAGE_SIZE_FINAL,
    "classes": ["person"],
    "detection_scales": ["P2/4", "P3/8", "P4/16", "P5/32"],
    "training_stages": {
        "stage1": {"epochs": STAGE_1_EPOCHS, "checkpoint": str(STAGE_1_BEST)},
        "stage2": {"epochs": STAGE_2_EPOCHS, "checkpoint": str(STAGE_2_BEST)},
        "stage3": {"epochs": STAGE_3_EPOCHS, "checkpoint": str(STAGE_3_BEST)},
    },
    "environment": {
        "python": sys.version,
        "ultralytics": ultralytics.__version__,
        "torch": torch.__version__,
        "cuda_runtime": torch.version.cuda,
        "gpu": GPU_NAME,
        "gpu_memory_gb": GPU_MEMORY_GB,
    },
}

metadata_path = FINAL_DIR / "model_metadata.json"
metadata_path.write_text(
    json.dumps(model_metadata, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

hash_path = FINAL_DIR / "SHA256.txt"
hash_path.write_text(f"{FINAL_SHA256}  {FINAL_PT.name}\n", encoding="utf-8")

print("Final PT:", FINAL_PT)
print(f"Size: {FINAL_SIZE_MB:.2f} MB")
print("SHA256:", FINAL_SHA256)

del final_model
clear_memory()

In [ ]:
# ============================================================
# CELL 12 — One-time final evaluation on Test
# ============================================================

def speed_value(metrics: Any, key: str) -> float:
    return float(metrics.speed.get(key, float("nan")))


test_model = YOLO(str(FINAL_PT))

test_metrics = test_model.val(
    data=str(FULL_DATA_YAML),
    split="test",
    imgsz=IMAGE_SIZE_FINAL,
    batch=BATCHES["val"],
    device=DEVICE,
    workers=WORKERS,
    single_cls=True,
    conf=TEST_CONF,
    iou=VAL_IOU,
    max_det=MAX_DET,
    plots=True,
    verbose=True,
    exist_ok=True,
    project=str(REPORTS_DIR / "validation"),
    name="final_test_yolo26s_p2",
)

final_test_row = {
    "model": "YOLO26s-P2 final",
    "checkpoint": str(FINAL_PT),
    "precision": float(test_metrics.box.mp),
    "recall": float(test_metrics.box.mr),
    "map50": float(test_metrics.box.map50),
    "map50_95": float(test_metrics.box.map),
    "preprocess_ms": speed_value(test_metrics, "preprocess"),
    "inference_ms": speed_value(test_metrics, "inference"),
    "postprocess_ms": speed_value(test_metrics, "postprocess"),
}

final_test_df = pd.DataFrame([final_test_row])
final_test_csv = REPORTS_DIR / "final_test_results.csv"
final_test_df.to_csv(final_test_csv, index=False)

display(final_test_df)
print("Final test report:", final_test_csv)

del test_model
clear_memory()

In [ ]:
# ============================================================
# CELL 13 — Optional descriptive baseline comparison
# ============================================================

comparison_rows = [final_test_row]

if BASELINE_PT is not None:
    baseline_path = Path(BASELINE_PT)
    if baseline_path.exists():
        baseline_model = YOLO(str(baseline_path))
        baseline_metrics = baseline_model.val(
            data=str(FULL_DATA_YAML),
            split="test",
            imgsz=IMAGE_SIZE_FINAL,
            batch=BATCHES["val"],
            device=DEVICE,
            workers=WORKERS,
            single_cls=True,
            conf=TEST_CONF,
            iou=VAL_IOU,
            max_det=MAX_DET,
            plots=False,
            verbose=True,
            exist_ok=True,
            project=str(REPORTS_DIR / "validation"),
            name="external_baseline_test",
        )
        comparison_rows.insert(0, {
            "model": "External baseline",
            "checkpoint": str(baseline_path),
            "precision": float(baseline_metrics.box.mp),
            "recall": float(baseline_metrics.box.mr),
            "map50": float(baseline_metrics.box.map50),
            "map50_95": float(baseline_metrics.box.map),
            "preprocess_ms": speed_value(baseline_metrics, "preprocess"),
            "inference_ms": speed_value(baseline_metrics, "inference"),
            "postprocess_ms": speed_value(baseline_metrics, "postprocess"),
        })
        del baseline_model
        clear_memory()
    else:
        warnings.warn(f"Baseline checkpoint not found: {baseline_path}")

comparison_df = pd.DataFrame(comparison_rows)
comparison_csv = REPORTS_DIR / "model_comparison_test.csv"
comparison_df.to_csv(comparison_csv, index=False)
display(comparison_df)

In [ ]:
# ============================================================
# CELL 14 — Custom size recall diagnostic at IoU=0.50
# This is NOT official COCO AP_small.
# ============================================================

def read_yolo_labels(label_path: Path, image_width: int, image_height: int) -> np.ndarray:
    if not label_path.exists() or label_path.stat().st_size == 0:
        return np.zeros((0, 4), dtype=np.float32)

    rows = []
    for line in label_path.read_text(encoding="utf-8").splitlines():
        values = line.strip().split()
        if len(values) < 5:
            continue
        _, cx, cy, width, height = map(float, values[:5])
        rows.append([
            (cx - width / 2) * image_width,
            (cy - height / 2) * image_height,
            (cx + width / 2) * image_width,
            (cy + height / 2) * image_height,
        ])
    return np.asarray(rows, dtype=np.float32).reshape(-1, 4)


def box_iou_matrix(a: np.ndarray, b: np.ndarray) -> np.ndarray:
    if len(a) == 0 or len(b) == 0:
        return np.zeros((len(a), len(b)), dtype=np.float32)

    x1 = np.maximum(a[:, None, 0], b[None, :, 0])
    y1 = np.maximum(a[:, None, 1], b[None, :, 1])
    x2 = np.minimum(a[:, None, 2], b[None, :, 2])
    y2 = np.minimum(a[:, None, 3], b[None, :, 3])

    intersection = np.maximum(0, x2 - x1) * np.maximum(0, y2 - y1)
    area_a = np.maximum(0, a[:, 2] - a[:, 0]) * np.maximum(0, a[:, 3] - a[:, 1])
    area_b = np.maximum(0, b[:, 2] - b[:, 0]) * np.maximum(0, b[:, 3] - b[:, 1])
    union = area_a[:, None] + area_b[None, :] - intersection
    return intersection / np.maximum(union, 1e-9)


def greedy_match(gt_boxes, pred_boxes, pred_conf, threshold=0.50):
    if len(gt_boxes) == 0 or len(pred_boxes) == 0:
        return set(), {}

    order = np.argsort(-pred_conf)
    pred_boxes = pred_boxes[order]
    ious = box_iou_matrix(pred_boxes, gt_boxes)

    matched_gt = set()
    matched_iou = {}

    for pred_index in range(len(pred_boxes)):
        gt_index = int(np.argmax(ious[pred_index]))
        value = float(ious[pred_index, gt_index])
        if value >= threshold and gt_index not in matched_gt:
            matched_gt.add(gt_index)
            matched_iou[gt_index] = value

    return matched_gt, matched_iou


def size_category(box, image_width, image_height, target_size=1280):
    scale = min(target_size / image_width, target_size / image_height)
    width = max(0.0, box[2] - box[0]) * scale
    height = max(0.0, box[3] - box[1]) * scale
    area = width * height

    if area < 16**2:
        return "very_small_lt_16sq"
    if area < 32**2:
        return "small_16_to_32sq"
    if area < 96**2:
        return "medium_32_to_96sq"
    return "large_ge_96sq"


if RUN_SIZE_DIAGNOSTIC:
    diagnostic_model = YOLO(str(FINAL_PT))
    test_images = collect_images(resolve_split_entries(FULL_DATA_YAML, "test"))
    counts = defaultdict(lambda: {"gt": 0, "matched": 0, "iou_sum": 0.0})

    for image_path in tqdm(test_images, desc="Size diagnostic", unit="image"):
        image = cv2.imread(str(image_path))
        if image is None:
            continue

        image_height, image_width = image.shape[:2]
        gt_boxes = read_yolo_labels(
            image_to_label_path(image_path), image_width, image_height
        )

        result = diagnostic_model.predict(
            source=image,
            imgsz=IMAGE_SIZE_FINAL,
            conf=TEST_CONF,
            iou=VAL_IOU,
            max_det=MAX_DET,
            device=DEVICE,
            half=True,
            verbose=False,
        )[0]

        if result.boxes is None or len(result.boxes) == 0:
            pred_boxes = np.zeros((0, 4), dtype=np.float32)
            pred_conf = np.zeros((0,), dtype=np.float32)
        else:
            pred_boxes = result.boxes.xyxy.detach().cpu().numpy().astype(np.float32)
            pred_conf = result.boxes.conf.detach().cpu().numpy().astype(np.float32)

        matched_gt, matched_iou = greedy_match(
            gt_boxes, pred_boxes, pred_conf, threshold=0.50
        )

        for gt_index, gt_box in enumerate(gt_boxes):
            category = size_category(
                gt_box, image_width, image_height, IMAGE_SIZE_FINAL
            )
            counts[category]["gt"] += 1
            if gt_index in matched_gt:
                counts[category]["matched"] += 1
                counts[category]["iou_sum"] += matched_iou[gt_index]

    rows = []
    for category, values in counts.items():
        rows.append({
            "size_category": category,
            "ground_truth": values["gt"],
            "matched_iou50": values["matched"],
            "recall_iou50": (
                values["matched"] / values["gt"] if values["gt"] else np.nan
            ),
            "mean_iou_of_matches": (
                values["iou_sum"] / values["matched"]
                if values["matched"] else np.nan
            ),
        })

    size_df = pd.DataFrame(rows).sort_values("size_category")
    size_csv = REPORTS_DIR / "size_recall_diagnostic.csv"
    size_df.to_csv(size_csv, index=False)
    display(size_df)
    print("Custom diagnostic report:", size_csv)

    del diagnostic_model
    clear_memory()

In [ ]:
# ============================================================
# CELL 15 — Export ONNX FP32 and FP16
# ============================================================

def move_export(exported: str | Path, destination: Path) -> Path:
    source = Path(str(exported)).resolve()
    if not source.exists():
        raise FileNotFoundError(f"Exported file does not exist: {source}")
    if destination.exists():
        destination.unlink()
    shutil.move(str(source), str(destination))
    return destination


ONNX_FP32 = None
ONNX_FP16 = None

if RUN_ONNX_EXPORT:
    export_model = YOLO(str(FINAL_PT))
    exported_fp32 = export_model.export(
        format="onnx",
        imgsz=IMAGE_SIZE_FINAL,
        batch=1,
        device=DEVICE,
        half=False,
        dynamic=False,
        simplify=True,
        opset=17,
        nms=False,
    )
    ONNX_FP32 = move_export(
        exported_fp32,
        EXPORTS_DIR / "yolo26s_p2_person_1280_fp32.onnx",
    )

    export_model = YOLO(str(FINAL_PT))
    exported_fp16 = export_model.export(
        format="onnx",
        imgsz=IMAGE_SIZE_FINAL,
        batch=1,
        device=DEVICE,
        half=True,
        dynamic=False,
        simplify=True,
        opset=17,
        nms=False,
    )
    ONNX_FP16 = move_export(
        exported_fp16,
        EXPORTS_DIR / "yolo26s_p2_person_1280_fp16.onnx",
    )

    print("ONNX FP32:", ONNX_FP32)
    print("ONNX FP16:", ONNX_FP16)

    del export_model
    clear_memory()

In [ ]:
# ============================================================
# CELL 16 — Inspect and optionally validate ONNX exports
# ============================================================

import onnx


def inspect_onnx(path: Path) -> dict[str, Any]:
    graph = onnx.load(str(path))
    dtypes = Counter(
        onnx.TensorProto.DataType.Name(item.data_type)
        for item in graph.graph.initializer
    )
    return {
        "path": str(path),
        "size_mb": path.stat().st_size / 1024**2,
        "sha256": sha256_file(path),
        "initializer_dtypes": dict(dtypes),
    }


export_rows = [{
    "candidate": "pytorch_pt",
    "path": str(FINAL_PT),
    "format": "pt",
    "declared_precision": "runtime-controlled",
    "size_mb": FINAL_PT.stat().st_size / 1024**2,
    "sha256": sha256_file(FINAL_PT),
    **{
        key: final_test_row[key]
        for key in (
            "precision", "recall", "map50", "map50_95",
            "preprocess_ms", "inference_ms", "postprocess_ms"
        )
    },
    "error": "",
}]

for candidate, path, declared_precision in (
    ("onnx_fp32", ONNX_FP32, "fp32"),
    ("onnx_fp16", ONNX_FP16, "fp16"),
):
    if path is None:
        continue

    info = inspect_onnx(path)
    print(json.dumps(info, indent=2))

    row = {
        "candidate": candidate,
        "path": str(path),
        "format": "onnx",
        "declared_precision": declared_precision,
        "size_mb": info["size_mb"],
        "sha256": info["sha256"],
        "precision": np.nan,
        "recall": np.nan,
        "map50": np.nan,
        "map50_95": np.nan,
        "preprocess_ms": np.nan,
        "inference_ms": np.nan,
        "postprocess_ms": np.nan,
        "error": "",
    }

    if RUN_ONNX_VALIDATION:
        try:
            onnx_model = YOLO(str(path))
            metrics = onnx_model.val(
                data=str(FULL_DATA_YAML),
                split="test",
                imgsz=IMAGE_SIZE_FINAL,
                batch=1,
                device=DEVICE,
                workers=WORKERS,
                single_cls=True,
                conf=TEST_CONF,
                iou=VAL_IOU,
                max_det=MAX_DET,
                plots=False,
                verbose=True,
                exist_ok=True,
                project=str(REPORTS_DIR / "validation"),
                name=f"{candidate}_test",
            )
            row.update({
                "precision": float(metrics.box.mp),
                "recall": float(metrics.box.mr),
                "map50": float(metrics.box.map50),
                "map50_95": float(metrics.box.map),
                "preprocess_ms": speed_value(metrics, "preprocess"),
                "inference_ms": speed_value(metrics, "inference"),
                "postprocess_ms": speed_value(metrics, "postprocess"),
            })
            del onnx_model
            clear_memory()
        except Exception as exc:
            row["error"] = repr(exc)
            warnings.warn(f"Validation failed for {candidate}: {exc}")
            clear_memory()

    export_rows.append(row)

deployment_df = pd.DataFrame(export_rows)
pt_map = float(
    deployment_df.loc[
        deployment_df["candidate"] == "pytorch_pt", "map50_95"
    ].iloc[0]
)
deployment_df["map50_95_drop_vs_pt"] = pt_map - deployment_df["map50_95"]

deployment_csv = REPORTS_DIR / "deployment_benchmark.csv"
deployment_df.to_csv(deployment_csv, index=False)
display(deployment_df)

In [ ]:
# ============================================================
# CELL 17 — Final experiment manifest
# ============================================================

summary = {
    "created_at": datetime.now().isoformat(),
    "final_model": model_metadata,
    "datasets": {
        "full_data_yaml": str(FULL_DATA_YAML),
        "hybrid_data_yaml": str(HYBRID_DATA_YAML),
        "dataset_summary_csv": str(dataset_summary_path),
    },
    "training": {
        "planned_epochs": STAGE_1_EPOCHS + STAGE_2_EPOCHS + STAGE_3_EPOCHS,
        "stage1_best": str(STAGE_1_BEST),
        "stage2_best": str(STAGE_2_BEST),
        "stage3_best": str(STAGE_3_BEST),
        "batches": BATCHES,
        "seed": SEED,
        "optimizer": "AdamW",
        "amp": True,
    },
    "test": final_test_row,
    "exports": deployment_df.replace({np.nan: None}).to_dict(orient="records"),
}

summary_path = REPORTS_DIR / "final_experiment_summary.json"
summary_path.write_text(
    json.dumps(summary, indent=2, ensure_ascii=False, default=str),
    encoding="utf-8",
)

print("=" * 110)
print("PIPELINE COMPLETED")
print("Final PT:", FINAL_PT)
print("SHA256:", FINAL_SHA256)
print("Final test CSV:", final_test_csv)
print("Deployment benchmark:", deployment_csv)
print("Summary:", summary_path)
print("=" * 110)

## ساخت TensorRT روی RTX 3070

فایل TensorRT را روی همان رایانه مقصد و همان RTX 3070 بسازید:

```python
from ultralytics import YOLO

model = YOLO(r"D:\path\to\yolo26s_p2_person_1280_final_best.pt")

engine_path = model.export(
    format="engine",
    imgsz=1280,
    batch=1,
    device=0,
    half=True,
    dynamic=False,
    workspace=4,
    nms=False,
)

print(engine_path)
```

سپس Engine را با همان `data.yaml` و `split="test"` ارزیابی کنید و افت `mAP50-95` را با فایل PT مقایسه کنید.